In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [2]:
import os
from google.colab import userdata

os.environ['KAGGLE_TOKEN'] = userdata.get('KAGGLE_TOKEN')

!pip install --upgrade kaggle -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.9/257.9 kB 6.2 MB/s eta 0:00:00


In [3]:
import requests
import zipfile

token = userdata.get('KAGGLE_TOKEN')
headers = {"Authorization": f"Bearer {token}"}

print("Downloading...")
r = requests.get(
    "https://www.kaggle.com/api/v1/datasets/download/abhisheksjha/time-series-air-quality-data-of-india-2010-2023",
    headers=headers,
    stream=True
)
print(f"Status: {r.status_code}")

with open('/content/air_quality.zip', 'wb') as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)

print("Extracting...")
os.makedirs('/content/air_quality', exist_ok=True)
with zipfile.ZipFile('/content/air_quality.zip', 'r') as z:
    z.extractall('/content/air_quality')

print("Done!")

Downloading...
Status: 200
Extracting...
Done!


In [4]:
final_files = {
    'Delhi':     'DL030',
    'Mumbai':    'MH015',
    'Chennai':   'TN003',
    'Hyderabad': 'TG004',
    'Bengaluru': 'KA008',
}

kaggle_dfs = {}

for city, fname in final_files.items():
    df = pd.read_csv(f'/content/air_quality/{fname}.csv')
    pm25_col = [c for c in df.columns if 'PM2.5' in c][0]
    df = df[['From Date', pm25_col]].copy()
    df.columns = ['date', 'pm25']
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date']).set_index('date')
    daily = df['pm25'].resample('D').mean().reset_index()
    daily.columns = ['date', 'pm25']
    daily = daily[daily['date'] >= '2019-01-01'].reset_index(drop=True)
    kaggle_dfs[city] = daily

for city, df in kaggle_dfs.items():
    print(f"{city}: {len(df)} days | missing={df['pm25'].isna().sum()} | {df['date'].min().date()} to {df['date'].max().date()}")

Delhi: 1551 days | missing=1 | 2019-01-01 to 2023-03-31
Mumbai: 1381 days | missing=33 | 2019-06-20 to 2023-03-31
Chennai: 1551 days | missing=23 | 2019-01-01 to 2023-03-31
Hyderabad: 1551 days | missing=20 | 2019-01-01 to 2023-03-31
Bengaluru: 1551 days | missing=38 | 2019-01-01 to 2023-03-31


In [5]:
for city in kaggle_dfs:
    kaggle_dfs[city]['pm25'] = kaggle_dfs[city]['pm25'].interpolate(method='linear')
    remaining = kaggle_dfs[city]['pm25'].isna().sum()
    print(f"{city}: {remaining} nulls remaining after interpolation")

Delhi: 0 nulls remaining after interpolation
Mumbai: 0 nulls remaining after interpolation
Chennai: 0 nulls remaining after interpolation
Hyderabad: 0 nulls remaining after interpolation
Bengaluru: 0 nulls remaining after interpolation


In [7]:
def walk_forward_splits(df, min_train_days = 365, forecast_horizon = 30, step_size = 30):
  splits = []
  n = len(df)
  start = min_train_days

  while start + step_size <= n:
    train = df.iloc[:start].copy()
    test = df.iloc[start : start + forecast_horizon].copy()
    splits.append((train,test))
    start += step_size
  return splits


In [9]:
splits = walk_forward_splits(kaggle_dfs["Delhi"])
print(len(kaggle_dfs["Delhi"]))
print(f"Delhi: {len(splits)} folds")
print(f"First fold  — train: {splits[0][0]['date'].min().date()} to {splits[0][0]['date'].max().date()} ({len(splits[0][0])} days)")
print(f"            — test:  {splits[0][1]['date'].min().date()} to {splits[0][1]['date'].max().date()} ({len(splits[0][1])} days)")
print(f"Last fold   — train: {splits[-1][0]['date'].min().date()} to {splits[-1][0]['date'].max().date()} ({len(splits[-1][0])} days)")
print(f"            — test:  {splits[-1][1]['date'].min().date()} to {splits[-1][1]['date'].max().date()} ({len(splits[-1][1])} days)")

1551
Delhi: 39 folds
First fold  — train: 2019-01-01 to 2019-12-31 (365 days)
            — test:  2020-01-01 to 2020-01-30 (30 days)
Last fold   — train: 2019-01-01 to 2023-02-13 (1505 days)
            — test:  2023-02-14 to 2023-03-15 (30 days)


In [14]:
def mae(actual, predicted):
    return np.mean(np.abs(actual - predicted))

def rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2))

def naive_persistence_forecast(train, horizon):
  last_value = train["pm25"].iloc[-1]
  return np.full(horizon, last_value)

def naive_seasonal_forecast(train, test, horizon):
  forecasts = []
  for date in test["date"]:
    last_year_date = date - pd.DateOffset(years = 1)
    match = train[train["date"] == last_year_date]["pm25"]
    if len(match) > 0:
      forecasts.append(match.values[0])
    else:
      forecasts.append(train["pm25"].iloc[-1])
  return np.array(forecasts)

In [16]:
results = {}

for city, df in kaggle_dfs.items():
  splits = walk_forward_splits(df)
  persistence_maes, persistence_rmses = [], []
  seasonal_maes,    seasonal_rmses    = [], []

  for train, test in splits:
    actual = test["pm25"].values
    horizon = len(actual)

    pers_forecast = naive_persistence_forecast(train, horizon)
    persistence_maes.append(mae(actual, pers_forecast))
    persistence_rmses.append(rmse(actual, pers_forecast))

    seas_forecast = naive_seasonal_forecast(train, test, horizon)
    seasonal_maes.append(mae(actual, seas_forecast))
    seasonal_rmses.append(rmse(actual, seas_forecast))

  results[city] = {
      'persistence_mae':  np.mean(persistence_maes),
      'persistence_rmse': np.mean(persistence_rmses),
      'seasonal_mae':     np.mean(seasonal_maes),
      'seasonal_rmse':    np.mean(seasonal_rmses),
      }
print(f"{'City':<12} {'Pers MAE':>10} {'Pers RMSE':>10} {'Seas MAE':>10} {'Seas RMSE':>10}")
print("-" * 55)
for city, r in results.items():
    print(f"{city:<12} {r['persistence_mae']:>10.2f} {r['persistence_rmse']:>10.2f} {r['seasonal_mae']:>10.2f} {r['seasonal_rmse']:>10.2f}")

City           Pers MAE  Pers RMSE   Seas MAE  Seas RMSE
-------------------------------------------------------
Delhi             45.07      55.01      46.42      58.72
Mumbai            17.56      21.02      19.27      23.75
Chennai           12.81      15.86      18.92      24.24
Hyderabad         15.54      18.22      15.74      19.15
Bengaluru         11.52      15.62      13.08      17.96
